# NDWI Analysis: Ayapel, Colombia

## Sentinel-2 and Landsat 8 | 2016-2025

Annual NDWI composites (2016-2025) for Ayapel, Cordoba, from Sentinel-2 and
Landsat 8 Surface Reflectance, exported as 10-band GeoTIFF stacks in EPSG:9377.

Ayapel (Cordoba) was selected for its wetland system, suitable for evaluating
multi-temporal surface-water conditions with NDWI. Sentinel-2 and Landsat 8 were
processed independently with sensor-specific cloud masks, and both resulting
10-band stacks (2016-2025) were exported as GeoTIFF in EPSG:9377.

**Limitation:** Sentinel-2 Level-2A data is unavailable for 2016, so `NDWI_2016`
in the Sentinel-2 stack has no information.


In [1]:
import os

PROJ_PATH = "/opt/conda/envs/geoprocesamiento/share/proj"

os.environ["PROJ_DATA"] = PROJ_PATH
os.environ["PROJ_LIB"] = PROJ_PATH

print("PROJ_DATA:", os.environ["PROJ_DATA"])
print("PROJ_LIB:", os.environ["PROJ_LIB"])

PROJ_DATA: /opt/conda/envs/geoprocesamiento/share/proj
PROJ_LIB: /opt/conda/envs/geoprocesamiento/share/proj


In [2]:
import ee
import geemap
import geopandas as gpd

from pathlib import Path
from pyproj import CRS

In [3]:
# Authenticate and initialize Google Earth Engine

try:
    ee.Initialize()
    print("Google Earth Engine initialized successfully.")

except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("Google Earth Engine authenticated and initialized successfully.")

Enter verification code:  4/1ATsMZqBdr-uH3U65n7Ty2Hxk7EWdyTVHrAWIJf5EhKcC-we20YsXMB1WKzs



Successfully saved authorization token.
Google Earth Engine authenticated and initialized successfully.


In [6]:
# Root folder inside Docker
root_folder = Path("/workspace/Taller_1")

# Load Colombian municipalities and reproject to the official
# Colombian projected CRS
gdf = gpd.read_file(root_folder / "municipios_colombia.gpkg")
gdf_unico = gdf.to_crs(epsg=9377)
gdf_unico["AREA_KM2"] = gdf_unico.geometry.area / 1_000_000

print(f"Initial CRS: {gdf.crs}")
gdf_unico.head()

DataSourceError: /workspace/Taller_1/municipios_colombia.gpkg: No such file or directory

In [ ]:
# Select Ayapel municipality in Córdoba

gdf_muni = gdf_unico[
    (gdf_unico["MPIO_CNMBR"].str.upper() == "AYAPEL")
    & (gdf_unico["DPTO_CCDGO"].astype(str).str.zfill(2) == "23")
].copy()

if gdf_muni.empty:
    raise ValueError("Ayapel municipality was not found.")

gdf_muni[["DPTO_CCDGO", "MPIO_CNMBR", "AREA_KM2"]]

In [ ]:
# Convert municipality geometry to WGS84
gdf_muni_wgs84 = gdf_muni.to_crs(epsg=4326)
geojson_geom = gdf_muni_wgs84.geometry.iloc[0].__geo_interface__
ee_muni_geom = ee.Geometry(geojson_geom)

print("Ayapel geometry successfully converted to Earth Engine.")

In [ ]:
# Calculate centroid in projected CRS
centroid_point = gdf_muni.geometry.centroid.to_crs(epsg=4326).iloc[0]
centroid = [centroid_point.y, centroid_point.x]

print("Map centroid:", centroid)

In [ ]:
Map_AOI = geemap.Map(center=centroid, zoom=10)
Map_AOI.add_gdf(gdf_muni_wgs84, layer_name="Ayapel municipality")
Map_AOI

In [ ]:
def mask_scale_s2(image):
    """Mask clouds/invalid pixels with the Sentinel-2 Scene Classification
    Layer (SCL) and scale B3/B8 to Surface Reflectance."""

    scl = image.select("SCL")

    mask = (
        scl.neq(0)        # No data
        .And(scl.neq(1))  # Saturated / defective
        .And(scl.neq(3))  # Cloud shadow
        .And(scl.neq(8))  # Medium probability cloud
        .And(scl.neq(9))  # High probability cloud
        .And(scl.neq(10)) # Cirrus
        .And(scl.neq(11)) # Snow / ice
    )

    reflectance = image.select(["B3", "B8"]).multiply(0.0001).toFloat()

    return reflectance.updateMask(mask).copyProperties(
        image, ["system:time_start"]
    )


def mask_scale_l8(image):
    """Mask clouds/invalid pixels with QA_PIXEL/QA_RADSAT and apply the
    Landsat 8 Collection 2 Surface Reflectance scaling factors."""

    qa = image.select("QA_PIXEL")
    # Bits 0-5: fill, dilated cloud, cirrus, cloud, cloud shadow, snow
    qa_mask = qa.bitwiseAnd(0b111111).eq(0)
    saturation_mask = image.select("QA_RADSAT").eq(0)

    reflectance = (
        image.select(["SR_B3", "SR_B5"]).multiply(0.0000275).add(-0.2).toFloat()
    )

    return (
        reflectance.updateMask(qa_mask)
        .updateMask(saturation_mask)
        .copyProperties(image, ["system:time_start"])
    )

In [ ]:
def calculate_ndwi(image, green_band, nir_band):
    """Generic NDWI calculation, reused for any sensor: (Green - NIR) / (Green + NIR)."""

    green = image.select(green_band)
    nir = image.select(nir_band)
    denominator = green.add(nir)

    return (
        green.subtract(nir)
        .divide(denominator)
        .updateMask(denominator.neq(0))
        .rename("NDWI")
    )

In [ ]:
INDEX_NAME = "NDWI"

# One config per sensor instead of duplicated annual_s2_* / annual_l8_*
# functions. Add a new sensor by adding a new entry here.
SENSOR_CONFIGS = {
    "S2": {
        "collection_id": "COPERNICUS/S2_SR_HARMONIZED",
        "green_band": "B3",
        "nir_band": "B8",
        "mask_fn": mask_scale_s2,
        "cloud_field": "CLOUDY_PIXEL_PERCENTAGE",
        "cloud_thresh": 30,
        "scale": 10,
        "min_year": 2017,  # S2 SR is not available for 2016
    },
    "L8": {
        "collection_id": "LANDSAT/LC08/C02/T1_L2",
        "green_band": "SR_B3",
        "nir_band": "SR_B5",
        "mask_fn": mask_scale_l8,
        "cloud_field": "CLOUD_COVER",
        "cloud_thresh": 30,
        "scale": 30,
        "min_year": 2016,
    },
}


def annual_index(year, cfg, aoi):
    """Build one annual median-composite index image for a given sensor config."""

    band_name = f"{INDEX_NAME}_{year}"

    if year < cfg["min_year"]:
        empty_image = (
            ee.Image.constant(0)
            .toFloat()
            .rename(band_name)
            .updateMask(ee.Image.constant(0))
            .clip(aoi)
        )
        return empty_image.set(
            {"year": year, "scene_count": 0, "status": "No data available"}
        )

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")  # filterDate() end is exclusive

    collection = (
        ee.ImageCollection(cfg["collection_id"])
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt(cfg["cloud_field"], cfg["cloud_thresh"]))
        .map(cfg["mask_fn"])
    )

    composite = collection.median()
    index_image = (
        calculate_ndwi(composite, cfg["green_band"], cfg["nir_band"])
        .rename(band_name)
        .clip(aoi)
    )

    return index_image.set({"year": year, "scene_count": collection.size()})

In [ ]:
years = list(range(2016, 2026))

annual_images = {
    sensor: [annual_index(year, cfg, ee_muni_geom) for year in years]
    for sensor, cfg in SENSOR_CONFIGS.items()
}

print("Annual image objects successfully created for:", list(annual_images.keys()))

In [ ]:
# Single getInfo() call for all sensors/years instead of one call per
# year-sensor combination (20 round-trips in the original version).
scene_counts = ee.Dictionary(
    {
        sensor: ee.List(images).map(lambda img: ee.Image(img).get("scene_count"))
        for sensor, images in annual_images.items()
    }
).getInfo()

for sensor, counts in scene_counts.items():
    print(sensor)
    print("-" * 30)
    for year, count in zip(years, counts):
        print(f"{year}: {count} scenes")
    print()

In [ ]:
def build_stack(images):
    """Combine a list of single-band annual images into one multi-band ee.Image."""

    stack = ee.Image(images[0])
    for image in images[1:]:
        stack = stack.addBands(image)
    return stack.toFloat()


stacks = {sensor: build_stack(images) for sensor, images in annual_images.items()}

s2_ndwi_stack = stacks["S2"]
l8_ndwi_stack = stacks["L8"]

print("Sentinel-2 and Landsat 8 stacks created.")

In [ ]:
for sensor, stack in stacks.items():
    print(f"{sensor} bands:", stack.bandNames().getInfo())

In [ ]:
import ipywidgets as widgets
from IPython.display import display

ndwi_vis = {
    "min": -0.5,
    "max": 0.8,
    "palette": ["brown", "yellow", "white", "lightblue", "blue", "darkblue"],
}

Map_NDWI_years = geemap.Map(center=centroid, zoom=10)
Map_NDWI_years.addLayer(ee_muni_geom, {"color": "black"}, "Ayapel boundary")

sensor_dropdown = widgets.Dropdown(options=list(stacks.keys()), value="S2", description="Sensor:")
year_dropdown = widgets.Dropdown(options=years, value=years[-1], description="Year:")


def update_layer(change=None):


    sensor = sensor_dropdown.value
    year = year_dropdown.value
    band_name = f"{INDEX_NAME}_{year}"

    # Remove the previous NDWI layer (if any) before adding the new one
    existing_layer = Map_NDWI_years.find_layer("NDWI")
    if existing_layer is not None:
        Map_NDWI_years.remove_layer(existing_layer)

    Map_NDWI_years.addLayer(
        stacks[sensor].select(band_name), ndwi_vis, "NDWI"
    )


sensor_dropdown.observe(update_layer, names="value")
year_dropdown.observe(update_layer, names="value")

update_layer()  # draw the initial layer (S2, last year)

display(widgets.HBox([sensor_dropdown, year_dropdown]))
Map_NDWI_years

In [ ]:
# Build EPSG:9377 as WKT because Earth Engine does not parse the
# EPSG:9377 authority code directly
crs_9377_wkt = CRS.from_epsg(9377).to_wkt(version="WKT1_GDAL")
ee.Projection(crs_9377_wkt).getInfo()

In [ ]:
export_tasks = {}

for sensor, stack in stacks.items():
    description = f"{sensor}_NDWI_Ayapel_2016_2025"
    task = ee.batch.Export.image.toDrive(
        image=stack,
        description=description,
        folder="GEE_EXPORTS_GEOPRO",
        fileNamePrefix=description,
        region=ee_muni_geom,
        crs=crs_9377_wkt,
        scale=SENSOR_CONFIGS[sensor]["scale"],
        maxPixels=1e13,
        fileFormat="GeoTIFF",
    )
    task.start()
    export_tasks[sensor] = task
    print(f"{description} export started.")

In [ ]:
for task in ee.batch.Task.list():
    status = task.status()
    print("=" * 60)
    print("Task:", status.get("description"))
    print("State:", status.get("state"))
    print("Error:", status.get("error_message", "No error message"))

In [ ]:
from pyproj import CRS

# Build EPSG:9377 as WKT because Earth Engine
# does not parse the EPSG:9377 authority code directly
crs_9377_wkt = CRS.from_epsg(9377).to_wkt(
    version="WKT1_GDAL"
)

print(crs_9377_wkt)

In [ ]:
test_projection = ee.Projection(crs_9377_wkt)

print(test_projection.getInfo())

In [ ]:
s2_export = ee.batch.Export.image.toDrive(
    image=s2_ndwi_stack,
    description="S2_NDWI_Ayapel_2016_2025",
    folder="GEE_EXPORTS_GEOPRO",
    fileNamePrefix="S2_NDWI_Ayapel_2016_2025",
    region=ee_muni_geom,
    crs=crs_9377_wkt,
    scale=10,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

s2_export.start()

print("Sentinel-2 NDWI export iniciada.")

In [ ]:
l8_export = ee.batch.Export.image.toDrive(
    image=l8_ndwi_stack,
    description="L8_NDWI_Ayapel_2016_2025",
    folder="GEE_EXPORTS_GEOPRO",
    fileNamePrefix="L8_NDWI_Ayapel_2016_2025",
    region=ee_muni_geom,
    crs=crs_9377_wkt,
    scale=30,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

l8_export.start()

print("Landsat 8 NDWI export iniciada.")

## Results download

The exported stacks (10 bands, EPSG:9377, GeoTIFF) are available for download from Google Drive:

- **Landsat 8** (`L8_NDWI_Ayapel_2016_2025.tif`): [Download](https://drive.google.com/file/d/1V2r9KveH-d9UEGniNVT7NWaxvicK9YMK/view?usp=drive_link)
- **Sentinel-2** (`S2_NDWI_Ayapel_2016_2025.tif`): [Download](https://drive.google.com/file/d/15H8FWJG04SSIF5ONoA1XzDJzcfsR3VpV/view?usp=drive_link)